# Data Pipeline
This notebook loads the raw dataset, explores it, and prepares it for model training.

In [ ]:
# pandas is the main library for loading and manipulating tabular data
# train_test_split divides the dataset into training and test sets
# StandardScaler normalizes feature values to have mean=0 and std=1,
# which is important for distance-based and linear models
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

### Load Data

In [ ]:
# reading the dataset from the CSV file into a DataFrame
# shape tells us the number of rows (patients) and columns (features)
# head() shows the first 5 rows so we can verify the data loaded correctly
df = pd.read_csv('../data/alzheimers_disease_data.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# info() shows the data type of each column and how many non-null values it has
# this helps identify columns with missing values or incorrect types before modeling
df.info()

In [ ]:
# describe() gives summary statistics: mean, std, min, max, and quartiles for every column
# include='all' ensures categorical columns are included as well
# .T transposes the result so each row is a feature, which is easier to read
df.describe(include='all').T

In [ ]:
# checking how balanced the target variable is
# 0 = no Alzheimer's diagnosis, 1 = diagnosed
# normalize=True returns proportions instead of raw counts
# if one class dominates heavily, the model might just always predict that class
print(df['Diagnosis'].value_counts(normalize=True).round(3))

### Preprocessing

In [ ]:
# PatientID is just an arbitrary row number with no predictive value
# DoctorInCharge is the same string for every row, so it adds no information
DROP_COLS = ['PatientID', 'DoctorInCharge']
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

# y is the target variable (what we want to predict)
# X contains all remaining columns that will be used as input features
y = df['Diagnosis'].astype(int)
X = df.drop(columns=['Diagnosis'])

# machine learning models require numeric input
# get_dummies converts categorical columns into binary (0/1) indicator columns
# drop_first=True removes one dummy per category to avoid multicollinearity
X = pd.get_dummies(X, drop_first=True)

print('Features shape:', X.shape)

In [ ]:
# splitting the dataset into 80% training and 20% test
# stratify=y ensures both splits preserve the original class distribution
# random_state=42 makes the split reproducible across runs
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# StandardScaler standardizes features to zero mean and unit variance
# fit_transform on training data computes the mean/std and applies the transformation
# transform on test data applies the same mean/std — fitting on test data would cause data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('X_train:', X_train.shape, '| X_test:', X_test.shape)
print('Train target dist:', y_train.value_counts(normalize=True).round(3).to_dict())